In [1]:
# %%
!pip install --upgrade onnx
!pip install --upgrade onnx onnxruntime
!pip install --upgrade onnx onnx-simplifier
!pip install --upgrade onnx onnx-simplifier onnxruntime


Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 117.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 120.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.5/323.5 kB 160.1 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.24.4
    Uninstalling protobuf-4.24.4:
      Successfully uninstalled protobuf-4.24.4
  Attempting uninstall: onnx
    Found existing installation: onnx 1.15.0rc2
    Uninstalling onnx-1.15.0rc2:
      Successfully uninstalled onnx-1.15.0rc2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf 24.2.0 requires protobuf<5,>=4.21, but you have protobuf 6.33.5 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 26.0
[notice] To update, run: p

In [2]:

# %%
import os
import random
import numpy as np
import onnx
from onnx import numpy_helper
import onnxruntime
from onnxsim import simplify
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantType, QuantFormat, quant_pre_process
from onnxruntime.quantization.calibrate import CalibrationMethod
from PIL import Image
from torchvision import transforms


In [3]:

# %%
BASE_DIR = os.getcwd()
INPUT_MODEL_PATH = os.path.join(BASE_DIR, "weights", "yolop-640-640-normalized.onnx")
OUTPUT_DIR = os.path.join(BASE_DIR, "weights")
OUTPUT_MODEL_PATH = os.path.join(OUTPUT_DIR, "yolop-640-640-normalized-quantized-int8-Entropy.onnx")
CALIBRATION_IMG_DIR = os.path.join(BASE_DIR, "dataset", "images", "val")
CALIBRATION_SIZE = 50
TARGET_SIZE = (640, 640)
PRIORITY_IMAGES = ["b1c66a42-6f7d68ca.jpg"]
USE_PER_CHANNEL = True
ENABLE_REDUCE_RANGE = True
CALIBRATION_METHOD = CalibrationMethod.Entropy


In [4]:

# %%
class YOLOPDataReader(CalibrationDataReader):
    
    def __init__(self, image_folder, model_path, size_limit=100, priority_list=None):
        self.image_folder = image_folder
        self.enum_data_dicts = None
        self.datasize = 0
        session = onnxruntime.InferenceSession(model_path, providers=['CPUExecutionProvider'])
        self.input_name = session.get_inputs()[0].name
        self.normalize = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        all_files = [f for f in os.listdir(image_folder) if f.endswith(('.jpg', '.png', '.jpeg'))]
        selected_files = []
        
        if priority_list:
            for img_name in priority_list:
                if img_name in all_files:
                    selected_files.append(img_name)
                else:
                    print(f"Image prioritaire non trouvee : {img_name}")
        
        if size_limit is None:
            self.image_files = selected_files
        else:
            remaining_slots = size_limit - len(selected_files)
            if remaining_slots > 0:
                pool = list(set(all_files) - set(selected_files))
                if len(pool) >= remaining_slots:
                    random_fill = random.sample(pool, remaining_slots)
                    selected_files.extend(random_fill)
                else:
                    selected_files.extend(pool)
            self.image_files = selected_files[:size_limit]
        
        self.datasize = len(self.image_files)
        print(f"DataReader : {self.datasize} images de calibration (Dont {len(priority_list or [])} prioritaires)")
    
    def letterbox_image(self, image, size):
        iw, ih = image.size
        w, h = size
        scale = min(w/iw, h/ih)
        nw = int(iw * scale)
        nh = int(ih * scale)
        
        image = image.resize((nw, nh), Image.BICUBIC)
        new_image = Image.new('RGB', size, (114, 114, 114))
        new_image.paste(image, ((w - nw) // 2, (h - nh) // 2))
        return new_image
    
    def get_next(self):
        if self.enum_data_dicts is None:
            self.enum_data_dicts = iter(self.load_batch())
        try:
            return next(self.enum_data_dicts)
        except StopIteration:
            return None
    
    def load_batch(self):
        for filename in self.image_files:
            image_path = os.path.join(self.image_folder, filename)
            image = Image.open(image_path).convert("RGB")
            image = self.letterbox_image(image, TARGET_SIZE)
            
            input_tensor = self.normalize(image)
            input_numpy = input_tensor.unsqueeze(0).numpy()
            
            yield {self.input_name: input_numpy}


In [5]:

# %%
def get_bn_weight_names(model_path):
    model = onnx.load(model_path)
    bn_weights = []
    bn_nodes = []
    
    for node in model.graph.node:
        if node.op_type == 'BatchNormalization':
            bn_nodes.append(node.name)
    
    for initializer in model.graph.initializer:
        if '.bn.weight' in initializer.name or '.bn.bias' in initializer.name:
            bn_weights.append(initializer.name)
    
    print(f"{len(bn_weights)} poids BatchNorm identifies")
    print(f"{len(bn_nodes)} nodes BatchNormalization restantes")
    return bn_weights


In [6]:
def force_bn_fusion(model_path, output_path):
    print(f"  Tentative de fusion forcée avec shapes statiques...")
    model = onnx.load(model_path)
    
    # 1. Récupération du nom de l'entrée
    input_name = model.graph.input[0].name
    
    # 2. Définition stricte des dimensions (Batch=1, RGB=3, 640x640)
    # C'est LA clé : sans ça, onnxsim ne touche pas aux BatchNorms
    input_shapes = {input_name: [1, 3, 640, 640]}
    
    bn_count_before = sum(1 for node in model.graph.node if node.op_type == 'BatchNormalization')
    
    try:
        # 3. On lance la simplification en écrasant les shapes
        model_sim, check = simplify(model, overwrite_input_shapes=input_shapes)
        
        if check:
            bn_count_after = sum(1 for node in model_sim.graph.node if node.op_type == 'BatchNormalization')
            print(f"  Fusion BatchNorm : {bn_count_before} -> {bn_count_after} nodes restantes")
            
            if bn_count_after > 0:
                print("  ATTENTION : Des BatchNorms subsistent. Le modèle est peut-être complexe.")
            else:
                print("  SUCCÈS : Tous les BatchNorms ont été fusionnés.")
                
            onnx.save(model_sim, output_path)
            return output_path
        else:
            print(f"  Simplification échouée (Check invalide), utilisation du modèle original")
            return model_path
            
    except Exception as e:
        print(f"  Erreur lors de la fusion : {e}")
        return model_path

In [7]:

# %%
def quantize_model():
    print("=" * 70)
    print("  QUANTIFICATION YOLOP POUR DEPLOIEMENT NPU (INT8)")
    print("=" * 70)
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    if not os.path.exists(INPUT_MODEL_PATH):
        print(f"ERREUR : Modele introuvable : {INPUT_MODEL_PATH}")
        return
    
    print("\n[1/4] Simplification et fusion forcee Conv+BN...")
    SIMPLIFIED_PATH = os.path.join(OUTPUT_DIR, "temp_simplified.onnx")
    
    SIMPLIFIED_PATH = force_bn_fusion(INPUT_MODEL_PATH, SIMPLIFIED_PATH)
    
    print(f"Validation et pre-traitement ONNX Runtime...")
    PREPROCESSED_PATH = os.path.join(OUTPUT_DIR, "temp_preprocessed.onnx")
    quant_pre_process(SIMPLIFIED_PATH, PREPROCESSED_PATH)
    
    if os.path.exists(SIMPLIFIED_PATH) and SIMPLIFIED_PATH != INPUT_MODEL_PATH:
        os.remove(SIMPLIFIED_PATH)
        
    SIMPLIFIED_PATH = PREPROCESSED_PATH
    print(f"Modele pre-traite : {SIMPLIFIED_PATH}")
    
    print("\n[2/4] Verification des poids BatchNorm...")
    bn_weights = get_bn_weight_names(SIMPLIFIED_PATH)
    
    print(f"\n[3/4] Preparation de la calibration...")
    print(f"Source : {CALIBRATION_IMG_DIR}")
    
    if not os.path.exists(CALIBRATION_IMG_DIR):
        print(f"ERREUR : Dossier de calibration introuvable")
        return
    
    dr = YOLOPDataReader(CALIBRATION_IMG_DIR, SIMPLIFIED_PATH, size_limit=CALIBRATION_SIZE, priority_list=PRIORITY_IMAGES)
    
    print(f"\n[4/4] Quantification INT8...")
    print(f"Format       : QDQ")
    print(f"Activations  : QUInt8")
    print(f"Weights      : QInt8")
    print(f"Per-channel  : {'Active' if USE_PER_CHANNEL else 'Desactive (per-tensor)'}")
    print(f"Reduce range : {'Active' if ENABLE_REDUCE_RANGE else 'Desactive'}")
    print(f"Calibration  : {CALIBRATION_METHOD}")
    print(f"Exclusions   : Aucune (Toutes les nodes en INT8)")
    
    try:
        extra_options = {}
        extra_options['WeightSymmetric'] = True
        extra_options['ActivationSymmetric'] = False
        
        quantize_static(
            model_input=SIMPLIFIED_PATH,
            model_output=OUTPUT_MODEL_PATH,
            calibration_data_reader=dr,
            quant_format=QuantFormat.QDQ,
            activation_type=QuantType.QUInt8,
            weight_type=QuantType.QInt8,
            per_channel=USE_PER_CHANNEL,
            reduce_range=ENABLE_REDUCE_RANGE,
            calibrate_method=CALIBRATION_METHOD,
            extra_options=extra_options
        )
        
        print("\n" + "=" * 70)
        print("  RESULTATS DE LA QUANTIFICATION")
        print("=" * 70)
        
        if os.path.exists(OUTPUT_MODEL_PATH):
            size_original = os.path.getsize(INPUT_MODEL_PATH) / (1024 * 1024)
            size_quantized = os.path.getsize(OUTPUT_MODEL_PATH) / (1024 * 1024)
            reduction = size_original / size_quantized
            
            print(f"\nTaille des modeles :")
            print(f"    Original (FP32)  : {size_original:>8.2f} MB")
            print(f"    Quantized (INT8) : {size_quantized:>8.2f} MB")
            print(f"    Reduction        : {reduction:>8.2f}x")
            
            print(f"\nModele INT8 :")
            print(f"    {OUTPUT_MODEL_PATH}")
            
            print(f"\nCaracteristiques du modele :")
            print(f"    Format : ONNX QDQ (Quantize-DeQuantize)")
            print(f"    Precision : INT8")
            print(f"    Mode : Per-tensor (compatible BatchNorm)")
            print(f"    Nodes exclues : 0 (Quantification complete)")
            
            print(f"\nNettoyage des fichiers temporaires...")
            if os.path.exists(SIMPLIFIED_PATH):
                os.remove(SIMPLIFIED_PATH)
                print(f"    temp_preprocessed.onnx supprime")
            
            print("\n" + "=" * 70)
        else:
            print("Erreur : Le fichier de sortie n'a pas ete cree")
            
    except Exception as e:
        print(f"\nERREUR lors de la quantification :")
        print(f"    {str(e)}")
        import traceback
        traceback.print_exc()


In [8]:

# %%
if __name__ == "__main__":
    quantize_model()

  QUANTIFICATION YOLOP POUR DEPLOIEMENT NPU (INT8)

[1/4] Simplification et fusion forcee Conv+BN...
  Tentative de fusion forcée avec shapes statiques...
  Fusion BatchNorm : 12 -> 12 nodes restantes
  ATTENTION : Des BatchNorms subsistent. Le modèle est peut-être complexe.
Validation et pre-traitement ONNX Runtime...
Modele pre-traite : /workspace/weights/temp_preprocessed.onnx

[2/4] Verification des poids BatchNorm...
24 poids BatchNorm identifies
12 nodes BatchNormalization restantes

[3/4] Preparation de la calibration...
Source : /workspace/dataset/images/val
DataReader : 50 images de calibration (Dont 1 prioritaires)

[4/4] Quantification INT8...
Format       : QDQ
Activations  : QUInt8
Weights      : QInt8
Per-channel  : Active
Reduce range : Active
Calibration  : CalibrationMethod.Entropy
Exclusions   : Aucune (Toutes les nodes en INT8)
Finding optimal threshold for each tensor using 'entropy' algorithm ...
Number of tensors : 536
Number of histogram bins : 128 (The number ma


  RESULTATS DE LA QUANTIFICATION

Taille des modeles :
    Original (FP32)  :    34.24 MB
    Quantized (INT8) :    11.83 MB
    Reduction        :     2.89x

Modele INT8 :
    /workspace/weights/yolop-640-640-normalized-quantized-int8-Entropy.onnx

Caracteristiques du modele :
    Format : ONNX QDQ (Quantize-DeQuantize)
    Precision : INT8
    Mode : Per-tensor (compatible BatchNorm)
    Nodes exclues : 0 (Quantification complete)

Nettoyage des fichiers temporaires...
    temp_preprocessed.onnx supprime

